## ADD Norm
- 在Transformer中，数据过Attention层和FFN层后，都会经过一个Add & Norm处理。
- 其中，Add为residule block（残差模块），数据在这里进行residule connection（残差连接）。
- 而Norm即为Normalization（标准化）模块。
- Transformer中采用的是Layer Normalization（层标准化）方式。
- https://www.zhihu.com/question/309177367

In [1]:
import torch
from transformers import BertTokenizer, BertModel

In [2]:
model_name = 'bert-base-uncased'

In [3]:
tokenizer = BertTokenizer.from_pretrained(model_name)

In [4]:
model = BertModel.from_pretrained(model_name, output_hidden_states=True)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [25]:
model.encoder.layer[0]

BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (query): Linear(in_features=768, out_features=768, bias=True)
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): Linear(in_features=768, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=3072, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): BertOutput(
    (dense): Linear(in_features=3072, out_features=768, bias=True)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

- BertLayer
    - attention: BertAttention
        - self: BertSelfAttention
        - output: BertSelfOutput
    - intermediate: BertIntermediate, 768=>4*768
    - output: BertOutput, 4*768 => 768

In [30]:
# 重点注意intermediate_size
model.config

BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_hidden_states": true,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.27.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

In [8]:
test_sent = 'this is a test sentence'


In [10]:
model_input = tokenizer(test_sent, return_tensors='pt')

In [11]:
model.eval()
with torch.no_grad():
    output = model(**model_input)

In [29]:
output.keys()

odict_keys(['last_hidden_state', 'pooler_output', 'hidden_states'])

In [12]:
# embeddings
output[2][0]

tensor([[[ 0.1686, -0.2858, -0.3261,  ..., -0.0276,  0.0383,  0.1640],
         [-0.6485,  0.6739, -0.0932,  ...,  0.4475,  0.6696,  0.1820],
         [-0.6270, -0.0633, -0.3143,  ...,  0.3427,  0.4636,  0.4594],
         ...,
         [ 0.6010, -0.6970, -0.2001,  ...,  0.2960,  0.2060, -1.7181],
         [ 0.8323,  0.2878,  0.0021,  ...,  0.2628, -1.1310, -1.2708],
         [-0.1481, -0.2948, -0.1690,  ..., -0.5009,  0.2544, -0.0700]]])

In [13]:
# first bert layer output
output[2][1]

tensor([[[ 0.1556, -0.0080, -0.0707,  ...,  0.0786,  0.0213,  0.0616],
         [-0.5333,  0.5799,  0.1044,  ...,  0.0241,  0.4888,  0.0161],
         [-1.0609, -0.3058, -0.5043,  ...,  0.1874,  0.2874,  0.4032],
         ...,
         [ 0.8206, -0.6656, -0.7054,  ...,  0.1347,  0.1117, -1.9040],
         [ 1.1128,  0.6603, -0.1509,  ...,  0.3253, -1.0006, -1.9106],
         [-0.0736,  0.0346,  0.0376,  ..., -0.4506,  0.6585, -0.0502]]])

In [14]:
embeddings = output[2][0]

In [15]:
layer = model.encoder.layer[0]

### 2.1 第一次 add & norm，发生在 mha 内部

In [18]:
mha_output = layer.attention.self(embeddings)
mha_output

(tensor([[[ 0.2979,  0.0801, -0.0037,  ..., -0.0142,  0.1290,  0.0828],
          [ 0.3935,  0.1356, -0.0920,  ...,  0.0211,  0.1677,  0.0011],
          [ 0.1696,  0.1449, -0.1039,  ...,  0.1604,  0.2172,  0.0310],
          ...,
          [-0.0617,  0.1968, -0.0669,  ...,  0.1126,  0.1933, -0.0204],
          [-0.2835,  0.1495, -0.0021,  ...,  0.0973,  0.1865, -0.0636],
          [ 0.2575,  0.1120, -0.1008,  ...,  0.0175,  0.1508,  0.0878]]],
        grad_fn=<ViewBackward0>),)

In [27]:
len(mha_output)

1

In [19]:
attn_output = layer.attention.output(mha_output[0], embeddings)

In [20]:
mlp1 = layer.intermediate(attn_output)
mlp1

tensor([[[-6.5261e-03, -4.5375e-02, -1.3537e-02,  ..., -3.3436e-02,
          -1.1251e-02, -1.3499e-03],
         [-7.4728e-02, -3.0861e-02, -6.9444e-04,  ..., -2.4785e-03,
          -3.7003e-02, -1.0934e-03],
         [-2.1356e-02, -7.4825e-04, -1.7961e-05,  ..., -4.2701e-04,
          -3.1277e-02, -6.5075e-03],
         ...,
         [-1.5633e-01, -1.5660e-02, -1.7594e-02,  ..., -1.8444e-03,
          -1.9849e-03, -3.5249e-02],
         [-1.6379e-01, -2.3173e-02, -1.2510e-03,  ..., -2.8437e-03,
          -1.5373e-01, -2.3979e-02],
         [-1.1528e-03, -5.6150e-02, -2.2484e-03,  ..., -4.7186e-03,
          -1.8783e-02, -1.3366e-05]]], grad_fn=<GeluBackward0>)

In [21]:
mlp1.shape

torch.Size([1, 7, 3072])

In [22]:
mlp2 = layer.output(mlp1, attn_output)
mlp2

tensor([[[ 0.1556, -0.0080, -0.0707,  ...,  0.0786,  0.0213,  0.0616],
         [-0.5333,  0.5799,  0.1044,  ...,  0.0241,  0.4888,  0.0161],
         [-1.0609, -0.3058, -0.5043,  ...,  0.1874,  0.2874,  0.4032],
         ...,
         [ 0.8206, -0.6656, -0.7054,  ...,  0.1347,  0.1117, -1.9040],
         [ 1.1128,  0.6603, -0.1509,  ...,  0.3253, -1.0006, -1.9106],
         [-0.0736,  0.0346,  0.0376,  ..., -0.4506,  0.6585, -0.0502]]],
       grad_fn=<NativeLayerNormBackward0>)

In [23]:
mlp2.shape

torch.Size([1, 7, 768])